[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-02-api-core.ipynb#scrollTo=aa01bb02)

---
# Day 2 · API Core Mechanics
**certified-journeys / claude-api-certified** · System prompts, multi-turn & streaming

> **Goal for today:** Master the three levers that control every Claude interaction — system prompts, conversation history, and streaming — plus token counting for cost management.

In [ ]:
%pip install -q anthropic

In [ ]:
import os
import anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## Step 1 · System prompts

The system prompt is your primary tool for persistent persona and behavior control. It's sent once per conversation and applies to every response.

| Without system prompt | With system prompt |
|---|---|
| Claude is a general assistant | Claude plays a specific role |
| Tone varies by default | Tone is consistent and predictable |
| Output format is uncontrolled | Output format can be enforced |

In [ ]:
# Without system prompt
r_no_system = client.messages.create(
    model=MODEL,
    max_tokens=128,
    messages=[{"role": "user", "content": "What is a REST API?"}]
)

# With system prompt — persona + format control
r_with_system = client.messages.create(
    model=MODEL,
    max_tokens=128,
    system="You are a terse technical instructor. Explain concepts in bullet points only. Never use full sentences. Maximum 3 bullets per answer.",
    messages=[{"role": "user", "content": "What is a REST API?"}]
)

print("WITHOUT system prompt:")
print(r_no_system.content[0].text)
print("\nWITH system prompt:")
print(r_with_system.content[0].text)

**What just happened?**
- The system prompt completely changed the output structure — same question, different format.
- **System prompts persist** for all turns in a conversation. User messages do not override them.
- **Write system prompts like job descriptions:** define the role, the constraints, and the output format explicitly.
- Claude follows system prompt instructions even if the user tries to override them.

## Step 2 · Multi-turn conversations

The Claude API is stateless — you must send the full conversation history with every request. The `messages` list alternates between `user` and `assistant` roles.

```
messages = [
  {"role": "user",      "content": "turn 1"},
  {"role": "assistant", "content": "response 1"},
  {"role": "user",      "content": "turn 2"},
  ...  ← Claude generates next
]
```

In [ ]:
def chat_loop(system: str, initial_message: str, follow_ups: list[str]) -> list[dict]:
    """Run a multi-turn conversation and return the full history."""
    history = [{"role": "user", "content": initial_message}]

    for i, user_msg in enumerate([initial_message] + follow_ups):
        if i > 0:
            history.append({"role": "user", "content": user_msg})

        r = client.messages.create(
            model=MODEL,
            max_tokens=256,
            system=system,
            messages=history
        )

        assistant_text = r.content[0].text
        history.append({"role": "assistant", "content": assistant_text})

        print(f"User: {user_msg}")
        print(f"Claude: {assistant_text}\n")

    return history


history = chat_loop(
    system="You are a Python tutor. Keep answers to 2-3 sentences.",
    initial_message="What is a list comprehension?",
    follow_ups=[
        "Give me a simple example.",
        "How is it different from a for loop?"
    ]
)

print(f"Total turns in history: {len(history)}")

**What just happened?**
- **Claude has no memory** — the `history` list is the entire conversation context.
- Every follow-up sends all previous turns again. Token cost grows with conversation length.
- **The alternating user/assistant pattern is required** — sending two user turns in a row will cause an API error.
- For long conversations, consider summarizing earlier turns to manage token costs.

## Step 3 · Streaming responses

Streaming sends tokens as they're generated, dramatically improving perceived latency for the user. Use `client.messages.stream()` as a context manager.

In [ ]:
import sys

print("Streaming response: ", end="", flush=True)

full_text = ""
with client.messages.stream(
    model=MODEL,
    max_tokens=256,
    messages=[{"role": "user", "content": "Write a haiku about APIs."}]
) as stream:
    for text_chunk in stream.text_stream:
        print(text_chunk, end="", flush=True)
        full_text += text_chunk

print("\n")

# After the stream, get the final message for metadata
final_message = stream.get_final_message()
print(f"Stop reason: {final_message.stop_reason}")
print(f"Output tokens: {final_message.usage.output_tokens}")

**What just happened?**
- **`stream.text_stream`** yields text chunks as they arrive — no waiting for the full response.
- **`get_final_message()`** returns the complete message after streaming finishes — use it for `stop_reason` and `usage`.
- Cost is identical to non-streaming — streaming only changes when you receive the data.
- For web apps, stream to the client with server-sent events (SSE) to show text appearing in real time.

## Step 4 · Token counting before sending

The `count_tokens` endpoint lets you estimate cost and check if you'll exceed the model's context window **before** making the real call — no tokens are charged.

In [ ]:
HAIKU_INPUT_COST_PER_MTok = 0.80   # USD per million input tokens (as of 2025)
HAIKU_CONTEXT_WINDOW = 200_000

def estimate_cost_and_fit(
    system: str,
    messages: list[dict],
    max_response_tokens: int = 1024
) -> dict:
    """Count tokens and estimate cost before sending."""
    count = client.messages.count_tokens(
        model=MODEL,
        system=system,
        messages=messages
    )
    input_tokens = count.input_tokens
    estimated_cost = (input_tokens / 1_000_000) * HAIKU_INPUT_COST_PER_MTok
    fits = (input_tokens + max_response_tokens) <= HAIKU_CONTEXT_WINDOW

    return {
        "input_tokens": input_tokens,
        "estimated_input_cost_usd": round(estimated_cost, 6),
        "fits_in_context": fits,
        "context_used_pct": round(input_tokens / HAIKU_CONTEXT_WINDOW * 100, 2)
    }


# Test with a short conversation
result = estimate_cost_and_fit(
    system="You are a helpful assistant.",
    messages=[
        {"role": "user", "content": "Explain transformer architecture in detail."},
        {"role": "assistant", "content": "Transformers use self-attention..."},
        {"role": "user", "content": "What is multi-head attention specifically?"}
    ]
)

for k, v in result.items():
    print(f"{k}: {v}")

**What just happened?**
- `count_tokens()` is **free** — it only reads the prompt, never generates output.
- **Use this in pipelines** where context length might vary (e.g., RAG with variable-length retrieved chunks).
- If `fits_in_context` is False, you need to trim the conversation history or document chunks before sending.
- Token costs are asymmetric: input is cheaper than output for most models.

## Step 5 · Tune max_tokens, temperature, and stop_sequences

| Parameter | Effect | Default |
|---|---|---|
| `max_tokens` | Hard cap on output length | Required |
| `temperature` | 0 = deterministic, 1 = creative | 1.0 |
| `stop_sequences` | Stop generation when a string appears | None |

In [ ]:
# Temperature = 0: highly consistent, predictable output
r_low_temp = client.messages.create(
    model=MODEL,
    max_tokens=64,
    temperature=0,
    messages=[{"role": "user", "content": "Flip a coin: heads or tails?"}]
)

# stop_sequences: stop generation at a delimiter
r_stop = client.messages.create(
    model=MODEL,
    max_tokens=256,
    stop_sequences=["---"],
    messages=[{
        "role": "user",
        "content": "Write a short Python function, then write --- after it."
    }]
)

print("Low temperature (deterministic):")
print(r_low_temp.content[0].text)
print(f"Stop reason: {r_low_temp.stop_reason}")

print("\nWith stop_sequence:")
print(r_stop.content[0].text)
print(f"Stop reason: {r_stop.stop_reason}")  # should be 'stop_sequence'

**What just happened?**
- **`temperature=0`** makes output nearly deterministic — use for classification, extraction, and structured tasks.
- **`stop_sequences`** gives you precise output boundaries — critical for structured generation (JSON, code blocks).
- `stop_reason: 'stop_sequence'` confirms the delimiter was hit.
- **High temperature** (0.8-1.0) is better for creative writing and brainstorming where variety is desired.

In [ ]:
# Challenge: Build a persistent chatbot
# Implement ChatBot so the conversation below works correctly.

class ChatBot:
    def __init__(self, system: str):
        self.system = system
        self.history = []

    def send(self, user_message: str) -> str:
        # Your solution here
        # 1. Append user_message to self.history
        # 2. Call client.messages.create() with self.system and self.history
        # 3. Append the assistant response to self.history
        # 4. Return the assistant text
        pass


bot = ChatBot(system="You are a concise Python tutor. 1-2 sentences per answer.")
r1 = bot.send("What is a decorator?")
r2 = bot.send("Give me a one-line example.")

print("Turn 1:", r1)
print("Turn 2:", r2)
assert len(bot.history) == 4, "History should have 4 messages (2 user + 2 assistant)"
print("✓ Challenge passed!")

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| System prompt | Persistent behavior control — like a job description for Claude |
| Multi-turn | API is stateless — send full history every request |
| Streaming | `client.messages.stream()` — tokens arrive live, cost is identical |
| `count_tokens()` | Free pre-check for context fit and cost estimation |
| `temperature=0` | Deterministic output for structured tasks |
| `stop_sequences` | Precise output boundaries for structured generation |

> **Tip:** System prompts are your best lever for consistent behavior. Write them like a job description, not a single instruction.

---
## What's next
**Day 3** → Prompt Evaluation Systems: build test datasets, grade outputs, and run A/B prompt comparisons.

Mark Day 2 complete in your [tracker](../index.html).